Checking PSD of R for DepthAnything v2 on COCO dataset

PSD of R <=> eigenvalues of R >= 0

In [ ]:
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
import numpy as np
import cv2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git

In [ ]:
import sys
sys.path.append("Depth-Anything-V2")

In [ ]:
!ls

In [ ]:
import sys
import os

# Ensure the parent directory of 'depth_anything_v2' is in sys.path
# The 'Depth-Anything-V2' repository is expected to be in '/content/'
repo_root_path = os.path.abspath(os.path.join(os.getcwd(), "Depth-Anything-V2"))
if repo_root_path not in sys.path:
    sys.path.insert(0, repo_root_path)

from depth_anything_v2.dpt import DepthAnythingV2

In [ ]:
model = DepthAnythingV2("vits").to(device)
model.eval()

In [ ]:
!wget http://images.cocodataset.org/zips/test2017.zip

In [ ]:
!unzip -q test2017.zip

In [ ]:
test_dir = "test2017"
files = os.listdir(test_dir)
len(files)

In [ ]:
from torchvision.transforms import Compose

class NormalizeImage(object):

    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, sample):
        sample["image"] = (sample["image"] - self.mean) / self.std
        return sample

class Resize(object):

    def __init__(self, width, height, multiple_of=14):
        self.width, self.height = width, height
        self.multiple_of = multiple_of

    def constrain_to_multiple_of(self, x, min_val=0, max_val=None):
        y = (np.round(x / self.multiple_of) * self.multiple_of).astype(int)
        if max_val is not None and y > max_val:
            y = (np.floor(x / self.multiple_of) * self.multiple_of).astype(int)
        if y < min_val:
            y = (np.ceil(x / self.multiple_of) * self.multiple_of).astype(int)
        return y

    def get_size(self, width, height):
        # determine new height and width
        scale_height, scale_width = self.height / height, self.width / width
        # scale such that output size is lower bound
        if scale_width > scale_height:
            scale_height = scale_width # fit width
        else:
            scale_width = scale_height # fit height
        new_height = self.constrain_to_multiple_of(scale_height * height, min_val=self.height)
        new_width = self.constrain_to_multiple_of(scale_width * width, min_val=self.width)
        return (new_width, new_height)

    def __call__(self, sample):
        width, height = self.get_size(sample["image"].shape[1], sample["image"].shape[0])
        sample["image"] = cv2.resize(sample["image"], (width, height), interpolation=cv2.INTER_CUBIC) # resize sample
        return sample

class PrepareForNet(object):

    def __init__(self):
        pass

    def __call__(self, sample):
        image = np.transpose(sample["image"], (2, 0, 1))
        sample["image"] = np.ascontiguousarray(image).astype(np.float32)
        return sample

def preprocessing(raw_image, input_size=518):
    transform = Compose([
        Resize(width=input_size, height=input_size, multiple_of=14),
        NormalizeImage(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        PrepareForNet(),
    ])
    h, w = raw_image.shape[:2]
    image = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB) / 255.0
    image = transform({'image': image})['image']
    image = torch.from_numpy(image).unsqueeze(0)
    image = image.to(device)
    return image

In [ ]:
model.pretrained

In [ ]:
result = model.pretrained.get_intermediate_layers(preprocessing(cv2.imread(os.path.join(test_dir, files[0]))).to(device))

In [ ]:
result[0].shape

In [ ]:
from torch import Tensor
from typing import Literal

In [ ]:
format2dim = {
    "hwc": 2,
    "chw": 0,
    "nc": 1,
    "bhwc": 3,
    "bchw": 1,
    "bnc": 2,
}

In [ ]:
def fixingValue(
    features: Tensor, # (B,C,H,W), (B,H,W,C), (C,H,W), (H,W,C), (B,H*W,C), or (H*W,C) accordig to `data_format`
    data_format: Literal["hwc", "chw", "nc", "bhwc", "bchw", "bnc" ] = "chw",
    eps: float = 1e-5,
):
    dim = format2dim[data_format]
    value = features.norm(dim=dim).abs().max() + eps
    return value.max().item()

In [ ]:
def extendWithFix(
    features: Tensor, # (B,C,H,W), (B,H,W,C), (C,H,W), (H,W,C), (B,H*W,C), or (H*W,C) accordig to `data_format`
    data_format: Literal["hwc", "chw", "nc", "bhwc", "bchw", "bnc" ] = "chw",
    eps: float = 1e-5,
) -> Tensor:
    """Extend the cosine similarity features by adding a feature that ensures the ncut algorithm is applicable."""
    dim = format2dim[data_format]
    value = fixingValue(features, data_format, eps)
    shape = list(features.shape)
    shape[dim] = 1
    shape = tuple(shape)
    extra = torch.full(shape, value, device=features.device, dtype=features.dtype)
    return torch.cat([features, extra], dim=dim) # (B,C+1,H,W), (B,H,W,C+1), (B,N,C+1), (C+1,H,W), (H,W,C+1), or (N,C+1) accordig to `data_format`

In [ ]:
img = cv2.imread(os.path.join(test_dir, files[0]))
blob = preprocessing(img).to(device)
with torch.no_grad():
    result = model.pretrained.get_intermediate_layers(blob,[11])
    feats = result[0][0][1:]
    W = torch.matmul(feats, feats.T)
    print(W.min().item(),W.max().item())
    d = torch.matmul(feats,feats.sum(dim=0).unsqueeze(1))
    print(d.min().item(),d.max().item())

In [ ]:
feats.shape

In [ ]:
for i, f in enumerate(files):
    img = cv2.imread(os.path.join(test_dir, f))
    blob = preprocessing(img).to(device)
    with torch.no_grad():
        result = model.pretrained.get_intermediate_layers(blob,[11])
        feats = result[0][0][1:]
    W = torch.matmul(feats, feats.T)
    minW = W.min().item()
    if minW <= 0:
        print(i,'W',minW,"!!!!!" if minW <= 0 else "")
    d = torch.matmul(feats,feats.sum(dim=0).unsqueeze(1))
    mind = d.min().item()
    if mind <= 0:
        print(i,'d',mind,"!!!!!" if mind <= 0 else "")
    else:
        D = torch.diag(d.squeeze(-1))
        eigenvalues, eigenvectors = torch.lobpcg(A=D-W, k=2, B=D, largest=False)
        if eigenvalues[0] < -1e5:
            print(i,'eig',eigenvalues[0].item(),"!!!!!" if eigenvalues[0].item() <= 0 else "")
    if i == 100:
        break

The values of W might be negative, d might not be well-defined, and R might not be PSD.

In [ ]:
for i, f in enumerate(files):
    img = cv2.imread(os.path.join(test_dir, f))
    blob = preprocessing(img).to(device)
    with torch.no_grad():
        result = model.pretrained.get_intermediate_layers(blob,[11])
        feats0 = result[0][0][1:]
        feats = extendWithFix(feats0,data_format="nc")
    W = torch.matmul(feats, feats.T)
    minW = W.min().item()
    if minW <= 0:
        print(i,minW,"!!!!!" if minW <= 0 else "")
    d = torch.matmul(feats,feats.sum(dim=0).unsqueeze(1))
    mind = d.min().item()
    if mind <= 0:
        print(i,mind,"!!!!!" if mind <= 0 else "")
    D = torch.diag(d.squeeze(-1))
    eigenvalues, eigenvectors = torch.lobpcg(A=D-W, k=2, B=D, largest=False)
    if eigenvalues[0] < -1e5:
        print(i,eigenvalues[0],"!!!!!" if eigenvalues[0] <= 0 else "")
        break
    if i % 500 == 0:
        print('done',i,'/',len(files),minW,mind,eigenvalues[0].item(),eigenvalues[1].item())

However with the fix, the values of W are positive and d is well defined, and R is always PSD